In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot 
from tqdm import trange

DATA = open("shakespeare.txt", "r").read()
DEVICE = "cuda"
VOCAB = list(sorted(set(DATA)))
VOCAB_MAP = {v:k for k,v in enumerate(VOCAB)}
VOCAB_SIZE = len(VOCAB)

def encode(str):
    return torch.tensor([VOCAB_MAP[x] for x in str], device=DEVICE)

def decode(l):
    return ''.join(VOCAB[x] for x in l)

DATA_TRAIN = encode(DATA[:int(len(DATA)*0.9)])
DATA_TEST =  encode(DATA[int(len(DATA)*0.9):])

def get_batch(data, width, size):
    a = torch.randint(len(data)-width,(size,1),device=data.device)
    a = a + torch.arange(width+1,device=data.device)
    return data[a[:,:width]], data[a[:,1:]]

class MultiHeadAttention(nn.Module):
    def __init__(self, context, embeddings, headsize, drop_p = 0.0):
        super().__init__()
        assert embeddings % headsize == 0, "Embeddings should be multiple of headsize"
        self.headcount = embeddings // headsize
        self.headsize = headsize
        self.qkv = nn.Linear(embeddings, embeddings*3, bias=False)
        self.proj = nn.Linear(embeddings, embeddings)
        self.dropout = nn.Dropout(drop_p)
        #self.register_buffer("causal_map", torch.tril(torch.ones((context, context), dtype=torch.bool)), persistent=False)
        angs = torch.tensor([1000.0**(-2*i/headsize) for i in range(headsize//2)]) * torch.arange(context).view(-1,1)
        self.register_buffer("rope_cos", torch.cos(angs), persistent=False)
        self.register_buffer("rope_sin", torch.sin(angs), persistent=False)

    def rope(self, x):
        N = x.shape[2]
        xx,yy = x.chunk(2, dim=-1)
        return torch.cat(
            [
                xx * self.rope_cos[:N,:] - yy * self.rope_sin[:N,:],
                xx * self.rope_sin[:N,:] + yy * self.rope_cos[:N,:],
            ], dim=-1)
    
    def forward(self, x):
        B,T,E = x.shape
        H = self.headcount
        W = self.headsize
        q,k,v = self.qkv(x).reshape(B,T,3,H,W).permute(2,0,3,1,4) # B,H,T,W
        q = self.rope(q)
        k = self.rope(k)
        #a = q @ k.transpose(-2,-1) / W**0.5
        #a = a.masked_fill(~self.causal_map[:T,:T], float("-inf"))
        #a = F.softmax(a, dim=-1)
        #a = self.dropout(a)
        #a = a @ v 
        a = F.scaled_dot_product_attention(q,k,v, is_causal=True, dropout_p = self.dropout.p if self.training else 0.0)
        a = a.transpose(-3,-2).reshape(B,T,E)
        return self.proj(a)

class FeedForward(nn.Module):
    def __init__(self, embeddings):
        super().__init__()
        self.lin = nn.Linear(embeddings, embeddings*4)
        self.gelu = nn.GELU()
        self.proj = nn.Linear(4*embeddings, embeddings)
    
    def forward(self, x):
        return self.proj(self.gelu(self.lin(x)))

class AttentionBlock(nn.Module):
    def __init__(self, context, embeddings, headsize, drop_p = 0.0):
        super().__init__()
        self.layn1 = nn.LayerNorm(embeddings)
        self.layn2 = nn.LayerNorm(embeddings)
        self.drop = nn.Dropout(drop_p)
        self.mha = MultiHeadAttention(
            context=context, 
            embeddings=embeddings, 
            headsize=headsize, 
            drop_p=drop_p)
        self.ff = FeedForward(embeddings=embeddings)

    def forward(self, x):
        x = x + self.drop(self.mha(self.layn1(x)))
        x = x + self.drop(self.ff(self.layn2(x)))
        return x

class Transformer(nn.Module):
    def __init__(self, vocab_size, context, embeddings, blocks, headsize, drop_p = 0.0):
        super().__init__()
        self.context = context
        self.embedding = nn.Embedding(vocab_size, embeddings)
        #self.embedding_pos = nn.Embedding(context, embeddings)
        self.attention = nn.Sequential(*[
            AttentionBlock(
                context=context, 
                embeddings=embeddings, 
                headsize=headsize, 
                drop_p=drop_p  
            ) for _ in range(blocks)
        ])
        self.layn = nn.LayerNorm(embeddings)
        self.drop = nn.Dropout(drop_p)
        self.lmh = nn.Linear(embeddings, vocab_size, bias=False)
        self.embedding.weight = self.lmh.weight
        #self.register_buffer("pos_shifts", torch.arange(context), persistent=False)

    def forward(self, x, y=None):
        B,T = x.shape
        x = self.embedding(x)# + self.embedding_pos(self.pos_shifts[:T])
        x = self.drop(x)
        x = self.attention(x)
        x = self.layn(x)
        x = self.lmh(x)
        if y is None:
            return x
        else:
            return F.cross_entropy(x.view(B*T,-1), y.reshape(-1))

    def generate(self, limit, prompt=""):
        was_train = m.training
        m.eval()
        device = self.embedding.weight.device
        out = [ 0 ] if len(prompt)==0 else encode(prompt).aslist()
        for _ in range(limit):
            x = torch.tensor(out[-self.context:], device=device).view(1,-1)
            x = self(x)
            out.append(torch.multinomial(
                F.softmax(x[:,-1,:].view(-1), dim=-1), 
                num_samples=1))
        m.train(was_train)
        return decode(out[(1 if len(prompt)==0 else 0):])

def estimate(m):
    was_train = m.training
    m.eval()
    l_ts,l_tr = (
        m(*get_batch(data,m.context,100))
        for data in [DATA_TEST, DATA_TRAIN]
    )
    m.train(was_train)
    return (l_ts, l_tr)

def train(m,o,batch_size, iterations):
    pbar = trange(0, iterations, desc="Learning")
    for i in pbar:
        o.zero_grad()
        loss = m(*get_batch(DATA_TRAIN, m.context, batch_size))
        loss.backward()
        o.step()
        if i%100==99:
            l_ts,l_tr = estimate(m)
            pbar.set_postfix({
                "loss_test": f"{l_ts:.4f}",
                "loss_train": f"{l_tr:.4f}",
            })

m = Transformer(
    vocab_size=VOCAB_SIZE,
    context=64, 
    embeddings=128, 
    blocks=8,
    headsize=32, 
    drop_p=0.1  
).to(DEVICE)

o = optim.AdamW(m.parameters(), lr=1e-3)

print(f"Model Size = {sum(p.numel() for p in m.parameters()):,}")

train(m,o,256,1000)

print(m.generate(1000))


# 1.7166
# 1.6821
# 1.5712





Model Size = 1,591,680


Learning: 100%|██████████| 1000/1000 [01:00<00:00, 16.48it/s, loss_test=1.5196, loss_train=1.3667]




As if I'll have lived found up your own short
To love me wanded drift: this drunk bears
The wisdom as he islemity; but colour,
I'll fair be truthing, mock much, I waschant he
now offence want any men told a' beauticurated,
Which, even she is is a news now makes.

Provoked:
Thou soundst head a desert, give leaves as
I men the suffer'd, she but make gone, her s sleeps
At hand the Clarence's suoul; be not as I crown'd soon.

DUCHESS OF YORK:
Say doth repose and your grace:
Away, that thy wife is a weary cuptrise.

JULIET:
So, sir! for you, the nurse of your lady,
To creature stand aunt, and by that;
If madam, I what, an my royal displely
From tribunes! the world.

AUFIDIUS:
Why you well, rose be pack, safety and say
Unamonous viil or budy.

GMERGORY:
My lord, happined; Lord, graw your patience?

HENRY BOLINGBROKE:
I prince, the rage of worthy drunks by him.

MERCUTIO:
Bid thou negl'st in a lady.

BRUTUS:
Ay, but on the loud?

LUCIO:
My lord!

-MENENIUS:
Here is the earth in Juliet, yet 